# Mid-Atlantic Gas Merchant: storage-year LP
Load data → validate → solve → report. Run top to bottom (*Runtime → Run all* in Colab).

In [ ]:
# Setup: in Colab, clone the repo and install dependencies; locally, do nothing.
import os, sys, subprocess

REPO_URL = "https://github.com/robjiakai/MidAtlanticGasMerchant.git"
REPO_DIR = "MidAtlanticGasMerchant"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

sys.path.insert(0, os.path.abspath("src"))
print("Working directory:", os.getcwd())


Working directory: /Users/robli/Documents/Columbia/2026Fall/Optimization/MidAtlanticGasMerchant


## 1. Load and validate inputs

In [ ]:
import pandas as pd
from load_data import load_data
from validate import validate_inputs, validate_solution
from model import solve

data = load_data()
problems = validate_inputs(data)
assert not problems, "Fix input problems before solving"


## 2. Solve

In [ ]:
sol = solve(data, verbose=False)
print(sol["status"])
print(f"Total annual cost: ${sol['objective']:,.0f}")
pd.Series(sol["costs"]).map("${:,.0f}".format)


In [ ]:
problems = validate_solution(sol, data)

## 3. Monthly plan

In [ ]:
T, J = data["T"], data["J"]
plan = pd.DataFrame({
    "hub price":    [data["P"][t] for t in T],
    "hub buy":      [sol["x"][t] for t in T],
    **{f"inv {f}":  [sol["I"][f, t] for t in T] for f in data["F"]},
    **{f"spot {j}": [sol["u"][j, t] for t in T] for j in J},
}, index=T).round(0)
plan


## 4. Shadow prices
What one more unit of each bottleneck is worth. Compare capacity duals with the reservation charge (12 × R_l) and storage duals with a lease fee.

In [ ]:
kc = pd.DataFrame({
    "contracted k (Dth/d)": sol["k"],
    "value of +1 Dth/d ($/yr)": sol["duals"]["KC"],
    "reservation cost ($/yr)": {l: 12 * data["R"][l] for l in data["L"]},
})
delivered = pd.DataFrame({j: [sol["duals"]["DM"][j, t] for t in T] for j in J}, index=T)
display(kc.round(2)); display(delivered.round(2).rename_axis("marginal $/Dth"))


In [ ]:
import os
os.makedirs("outputs", exist_ok=True)
plan.to_csv("outputs/monthly_plan.csv")
